# Diffusion Longform Retool

This notebook launches a fresh diffusion run with two explicit goals:

- train against **style-swapped conditioning** instead of only same-style reconstruction
- train against **adjacent-chunk stability** so Lab 4 is no longer asked to solve a mismatch it never saw in training

The retooled loss stack keeps identity as a stabilizer, but shifts pressure toward target-style realization and recursive continuity.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib
import json
import os
import subprocess
import sys

def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / 'lab 3.1').exists() and (p / 'README.md').exists():
            return p
    raise RuntimeError('Could not locate repo root from current notebook cwd.')

REPO = find_repo_root()
SCRIPT_DIR = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import diffusion_longform_retool_train as retool
import diffusion_downloads_batch as ddb
import diffusion_longform_compare as dlc

importlib.reload(retool)
importlib.reload(ddb)
importlib.reload(dlc)

print(REPO)

In [ ]:
TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = REPO / 'lab 3.1' / 'outputs' / 'diffusion_longform_retool' / f'run_{TAG}'
CACHE_DIR = REPO / 'saves2' / 'lab3_diffusion' / 'run_d001' / 'cache'
BASE_CHECKPOINT = retool.resolve_default_base_checkpoint()

RUN_TRAIN = True
RUN_SWEEP = True
RUN_BATCH = True
RUN_LONGFORM = True

TRAIN_CFG = {
    'epochs': 8,
    'batch_size': 1,
    'grad_accum': 2,
    'max_frames': 256,
    'lr': 8e-5,
    'identity_weight': 1.0,
    'style_weight': 1.8,
    'anchor_weight': 0.55,
    'envelope_weight': 0.30,
    'continuity_weight': 0.65,
    'hf_penalty_weight': 0.20,
    'anchor_bins': 44,
    'hf_start_bin': 56,
    'overlap_frames': 40,
    'hf_margin': 0.06,
    'style_every_steps': 1,
    'style_batch_splits': 1,
    'max_batches_per_epoch': 0,
    'monitor_steps': 25,
    'device': 'auto',
}

print('OUT_DIR =', OUT_DIR)
print('BASE_CHECKPOINT =', BASE_CHECKPOINT)
print(json.dumps(TRAIN_CFG, indent=2))

In [ ]:
train_cmd = [
    sys.executable,
    str(SCRIPT_DIR / 'diffusion_longform_retool_train.py'),
    '--cache-dir', str(CACHE_DIR),
    '--out-dir', str(OUT_DIR),
]
if BASE_CHECKPOINT is not None:
    train_cmd += ['--bootstrap-checkpoint', str(BASE_CHECKPOINT)]
for key, value in TRAIN_CFG.items():
    cli_key = '--' + key.replace('_', '-')
    train_cmd += [cli_key, str(value)]

print(' '.join(map(str, train_cmd)))
train_log = OUT_DIR / 'train.log'

if RUN_TRAIN:
    env = dict(os.environ)
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with train_log.open('w', encoding='utf-8', errors='replace') as log:
        proc = subprocess.Popen(train_cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=env)
        ret = proc.wait()
    print(train_log)
    print(train_log.read_text(encoding='utf-8', errors='replace')[-12000:])
    if ret != 0:
        raise RuntimeError(f'Training failed with exit code {ret}. See {train_log}')
else:
    print('Set RUN_TRAIN = True to launch the retooled diffusion training run.')

In [ ]:
sweep_cmd = [
    sys.executable,
    str(REPO / 'lab 3' / 'run_lab3_realism_sweep.py'),
    'diffusion',
    '--run-dir', str(OUT_DIR),
    '--include-all-epochs',
    '--n-samples', '12',
    '--min-style-target-acc', '0.35',
    '--min-style-target-cos', '0.10',
    '--max-fad-mert', '32',
]
print(' '.join(map(str, sweep_cmd)))
sweep_log = OUT_DIR / 'realism_sweep.log'

if RUN_SWEEP:
    with sweep_log.open('w', encoding='utf-8', errors='replace') as log:
        proc = subprocess.Popen(sweep_cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
        ret = proc.wait()
    print(sweep_log)
    print(sweep_log.read_text(encoding='utf-8', errors='replace')[-12000:])
    if ret != 0:
        raise RuntimeError(f'Realism sweep failed with exit code {ret}. See {sweep_log}')
else:
    print('Set RUN_SWEEP = True after training finishes.')

In [ ]:
def checkpoint_panel(run_dir: Path):
    ckpt_dir = run_dir / 'checkpoints'
    panel = []
    preferred = ['best.pt', 'latest.pt', 'epoch_005.pt', 'epoch_003.pt', 'epoch_008.pt']
    seen = set()
    for name in preferred:
        p = ckpt_dir / name
        if p.exists():
            label = p.stem
            panel.append({'label': label, 'path': p})
            seen.add(p.resolve())
    for p in sorted(ckpt_dir.glob('epoch_*.pt')):
        if p.resolve() not in seen and len(panel) < 6:
            panel.append({'label': p.stem, 'path': p})
            seen.add(p.resolve())
    return panel

batch_cfg = ddb.DiffusionDownloadsBatchConfig(
    run_dir=OUT_DIR,
    cache_dir=CACHE_DIR,
    output_root=REPO / 'lab 3.1' / 'outputs' / 'diffusion_downloads_batch',
    n_clips=30,
    seconds=3.0,
    seed=328,
    snapshot_latest_checkpoint=False,
)
panel = checkpoint_panel(OUT_DIR)
print(panel)

if RUN_BATCH:
    batch_summary = ddb.run_multi_checkpoint_inference(batch_cfg, panel)
    print(json.dumps(batch_summary, indent=2, default=str))
else:
    print('Set RUN_BATCH = True after training finishes if you want a random Downloads checkpoint panel.')

In [ ]:
longform_cfg = dlc.DiffusionLongformCompareConfig(
    run_dir=OUT_DIR,
    cache_dir=CACHE_DIR,
    output_root=REPO / 'lab 3.1' / 'outputs' / 'diffusion_longform_compare',
    source_seconds=45.0,
    n_songs=2,
    targets_per_song=2,
    checkpoint_labels=['best', 'epoch_005'],
    ddim_steps=50,
    guidance_scale=1.75,
    style_strength=0.60,
    t_start=240,
    t_start_end=180,
    reanchor_every=4,
    reanchor_t_start=160,
    overlap_seconds=0.5,
    source_prefix_blend=0.45,
    source_mel_blend=0.10,
    hf_source_blend=0.18,
    mel_time_smooth=3,
    mel_freq_smooth=0,
    assemble_domain='mel',
    snapshot_latest_checkpoint=False,
)
print(longform_cfg)

if RUN_LONGFORM:
    longform_summary = dlc.run_compare_panel(longform_cfg)
    print(json.dumps(longform_summary, indent=2, default=str))
else:
    print('Set RUN_LONGFORM = True after training finishes if you want longform compare on the new run.')